In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
for item in os.listdir('/kaggle/input'):
    print(item)
    subpath = f'/kaggle/input/{item}'
    if os.path.isdir(subpath):
        for sub in os.listdir(subpath):
            print(f"  {sub}")

In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install -q -U \
    "protobuf>=5.29,<6.0" \
    "bitsandbytes>=0.45.0" \
    "transformers==4.46.3" \
    "peft==0.13.2" \
    "trl==0.12.1" \
    "accelerate==1.1.1" \
    "datasets==3.1.0" \
    "wandb" \
    "sentencepiece"

In [1]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

!pip install -q -U \
    "protobuf>=5.29,<6.0" \
    "bitsandbytes>=0.45.0" \
    "transformers==4.46.3" \
    "peft==0.13.2" \
    "trl==0.12.1" \
    "accelerate==1.1.1" \
    "datasets==3.1.0" \
    "wandb" \
    "sentencepiece"

In [2]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import warnings
warnings.filterwarnings("ignore", message=".*AccumulateGrad node's stream.*")

from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTConfig, SFTTrainer

print(f"PyTorch: {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch: 2.10.0+cu128
GPU: Tesla T4
VRAM: 15.64 GB


In [3]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: royanjalii2511 (royanjalii2511-athena-education) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
train_ds = load_from_disk("/kaggle/input/notebooks/anjaliii25/02-data-prep/train_ds")
val_ds = load_from_disk("/kaggle/input/notebooks/anjaliii25/02-data-prep/val_ds")

print(f"Train: {len(train_ds):,}")
print(f"Val: {len(val_ds):,}")
print(f"Sample text length: {len(train_ds[0]['text'])} chars")
print(f"Columns: {train_ds.column_names}")

Train: 19,600
Val: 400
Sample text length: 4244 chars
Columns: ['text']


In [5]:
import shutil

for path in ["/kaggle/working/train_ds_local", "/kaggle/working/val_ds_local"]:
    if os.path.exists(path):
        shutil.rmtree(path)

train_ds.save_to_disk("/kaggle/working/train_ds_local")
val_ds.save_to_disk("/kaggle/working/val_ds_local")

train_ds = load_from_disk("/kaggle/working/train_ds_local")
val_ds = load_from_disk("/kaggle/working/val_ds_local")

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,}")
print("Now loaded from writable location ✓")

Saving the dataset (0/1 shards):   0%|          | 0/19600 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/400 [00:00<?, ? examples/s]

Train: 19,600 | Val: 400
Now loaded from writable location ✓


In [6]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=False,
)

print(f"Model loaded. Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Model dtype: {model.dtype}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded. Memory: 0.71 GB
Model dtype: torch.float16


In [7]:
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 36,929,536 || all params: 1,580,643,840 || trainable%: 2.3364


In [8]:
TRAIN_SAMPLES = 10000
train_ds_subset = train_ds.shuffle(seed=42).select(range(TRAIN_SAMPLES))

print(f"Training on {len(train_ds_subset):,} examples (subsampled from {len(train_ds):,})")
total_steps = len(train_ds_subset) // 32
print(f"Expected total steps: {total_steps}")
print(f"Estimated time at 85 sec/step: {total_steps * 85 / 3600:.1f} hours")

Training on 10,000 examples (subsampled from 19,600)
Expected total steps: 312
Estimated time at 85 sec/step: 7.4 hours


In [9]:
import warnings
warnings.filterwarnings("ignore", message=".*AccumulateGrad node's stream.*")

training_args = SFTConfig(
    output_dir="/kaggle/working/qwen-math-sft",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=32,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_seq_length=4096,
    logging_steps=5,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    eval_strategy="no",
    fp16=True,
    report_to="wandb",
    run_name="qwen1.5b-math-sft-r32-4096-10k-v2",
    dataset_text_field="text",
    packing=False,
    dataloader_num_workers=2,
    remove_unused_columns=True,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds_subset,
    tokenizer=tokenizer,
)

print("Trainer initialized with eval DISABLED.")
print(f"Will train for {total_steps} steps, saving every 50.")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Trainer initialized with eval DISABLED.
Will train for 312 steps, saving every 50.


In [10]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
5,0.837200
10,0.759300
15,0.672900
20,0.667500
25,0.636700
30,0.604800
35,0.599100
40,0.591900
45,0.585700
50,0.596000


TrainOutput(global_step=312, training_loss=0.5579595160789979, metrics={'train_runtime': 30148.6956, 'train_samples_per_second': 0.332, 'train_steps_per_second': 0.01, 'total_flos': 2.1633196999508582e+17, 'train_loss': 0.5579595160789979, 'epoch': 0.9984})

In [11]:
trainer.save_model("/kaggle/working/qwen-math-sft-final")
tokenizer.save_pretrained("/kaggle/working/qwen-math-sft-final")

# Verify
import os
files = os.listdir("/kaggle/working/qwen-math-sft-final")
total_size = sum(
    os.path.getsize(os.path.join("/kaggle/working/qwen-math-sft-final", f)) 
    for f in files
) / 1e6
print(f"Final adapter saved")
print(f"Files: {files}")
print(f"Total size: {total_size:.1f} MB")

Final adapter saved
Files: ['tokenizer_config.json', 'training_args.bin', 'adapter_model.safetensors', 'special_tokens_map.json', 'adapter_config.json', 'tokenizer.json', 'vocab.json', 'merges.txt', 'added_tokens.json', 'README.md']
Total size: 163.7 MB


In [12]:
model.eval()

test_problems = [
    "If a train travels 60 km/h for 2 hours and then 80 km/h for 3 hours, what is the total distance traveled?",
    "Find all real solutions to x^2 - 5x + 6 = 0.",
    "A rectangle has length twice its width. If the perimeter is 30, find the area.",
]

for i, problem in enumerate(test_problems):
    print(f"\n{'='*80}")
    print(f"PROBLEM {i+1}: {problem}")
    print('='*80)
    
    messages = [{"role": "user", "content": problem}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=1500,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(response[:2500])
    if len(response) > 2500:
        print(f"\n... [truncated, total: {len(response)} chars]")


PROBLEM 1: If a train travels 60 km/h for 2 hours and then 80 km/h for 3 hours, what is the total distance traveled?


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


<think>
Okay, let's see. The problem says that a train travels at different speeds for different times and asks for the total distance it covers. Hmm, right. So I need to figure out how far the train goes when it goes 60 kilometers per hour for two hours, and then another three hours at 80 kilometers per hour. 

First, maybe I should break this down into parts. The first part: traveling at 60 km/h for 2 hours. To find the distance covered in that time, I remember that speed equals distance divided by time. So if I can calculate the distance using that formula, then add it to the second part where the speed is 80 km/h over 3 hours.

Let me start with the first segment. Speed is given as 60 km/h, and time is 2 hours. Plugging those numbers into the formula:

Distance = Speed × Time

So,

Distance1 = 60 km/h × 2 h

Calculating that: 60 multiplied by 2 is 120. Wait, but hold on. If they say "km/h", does that mean kilometers per hour or something else? Let me check the units again. The ques

In [13]:
import shutil
import os

# Create zip of the adapter
shutil.make_archive(
    "/kaggle/working/adapter_backup",
    'zip',
    "/kaggle/working/qwen-math-sft-final"
)

size_mb = os.path.getsize("/kaggle/working/adapter_backup.zip") / 1e6
print(f"✓ Backup zip created: /kaggle/working/adapter_backup.zip")
print(f"Size: {size_mb:.1f} MB")

✓ Backup zip created: /kaggle/working/adapter_backup.zip
Size: 140.7 MB


In [14]:
import shutil
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Total: {total/1e9:.1f} GB")
print(f"Used: {used/1e9:.1f} GB")
print(f"Free: {free/1e9:.1f} GB")


Total: 21.0 GB
Used: 1.3 GB
Free: 19.6 GB
